# create manual pairs

In [ ]:
import pandas as pd
#defendents_ethnicity
drugs = ["cannabis","cocaine","hashish"]
dfs = ["no_impute", "impute", "only_age"]
for df_type in dfs:
# Load data
    for drug in drugs:
        df = pd.read_csv(
            rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\{df_type}\{drug}.csv"
        )
    
        # Sort by multiple columns
        df_sorted = df.sort_values(
            by=['total_amount','laboratory','role','sold_to_agent',
                'importation_offence', 'production_offence', 'use_offence', 'self_use',
                'year','Court','district', 'Location', 'Judges_Ethnic',
                'marital_status', 'age', 'Date'],
            ascending=[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
        ).reset_index(drop=True)

        df = df_sorted.reset_index(drop=True)

        pairs = []

        TREATMENT_VALUE = "Not Jewish"
        CONTROL_VALUE = "Jewish"

        while len(df) > 1:
            # Find first treatment case
            treatment_rows = df[df["Defendants_Ethnic"] == TREATMENT_VALUE]
            
            if len(treatment_rows) == 0:
                break
            
            first_row = treatment_rows.iloc[0]
            
            # Find nearest control (anywhere in the dataframe, not just below)
            candidates = df[df["Defendants_Ethnic"] == CONTROL_VALUE]
            
            if len(candidates) == 0:
                # No controls left
                break
            
            # Find the closest control by index (nearest in sorted order)
            distances = abs(candidates.index - first_row.name)
            closest_idx = candidates.index[distances.argmin()]
            matched_row = df.loc[closest_idx]
            
            pairs.append({
                "id_treat": first_row["original_case"],
                "id_control": matched_row["original_case"]
            })
            # Remove both rows
            df = df.drop([first_row.name, closest_idx]).reset_index(drop=True)

        # Save result
        pairs_df = pd.DataFrame(pairs)
        pairs_df.to_csv(f"data/pairs/Defendants_Ethnic/{df_type}/{drug}.csv", index=False)

        print(f"Created {len(pairs_df)} Arab–Jewish pairs for {drug} ({df_type})")

In [ ]:
import pandas as pd
# age
drugs = ["cannabis","cocaine","hashish"]
dfs = ["no_impute", "impute", "only_age"]
for df_type in dfs:
    # Load data
    for drug in drugs:
        df = pd.read_csv(
            rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\{df_type}\{drug}.csv"
        )

        df = df.sort_values(by=['total_amount','laboratory','role','sold_to_agent',
                'importation_offence', 'production_offence', 'use_offence', 'self_use',
                'year','Court','district', 'Location','Judges_Ethnic', 'Defendants_Ethnic',
                'marital_status', 'Date'],
            ascending=[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
        ).reset_index(drop=True)    

        df = df.reset_index(drop=True)
        pairs = []

        TREATMENT_VALUE = "Young"
        CONTROL_VALUE = "Older"
        df["age_binary"] = df["age"].apply(lambda x: TREATMENT_VALUE if x <= 28 else CONTROL_VALUE)

        while len(df) > 1:
            # Find first treatment case
            treatment_rows = df[df["age_binary"] == TREATMENT_VALUE]
            
            if len(treatment_rows) == 0:
                break
            
            first_row = treatment_rows.iloc[0]
            
            # Find nearest control (anywhere in the dataframe, not just below)
            candidates = df[df["age_binary"] == CONTROL_VALUE]
            
            if len(candidates) == 0:
                # No controls left
                break
            
            # Find the closest control by index (nearest in sorted order)
            distances = abs(candidates.index - first_row.name)
            closest_idx = candidates.index[distances.argmin()]
            matched_row = df.loc[closest_idx]
            
            pairs.append({
                "id_treat": first_row["original_case"],
                "id_control": matched_row["original_case"]
            })
            
            # Remove both rows
            df = df.drop([first_row.name, closest_idx]).reset_index(drop=True)
        # Save result
        pairs_df = pd.DataFrame(pairs)
        pairs_df.to_csv(f"data/pairs/age/{df_type}/{drug}.csv", index=False)
        print(f"Created {len(pairs_df)} age-based pairs for {drug} ({df_type})")

In [ ]:
import pandas as pd
# year
drugs = ["cannabis","cocaine","hashish"]
dfs = ["no_impute", "impute", "only_age"]
options = ['corona_period', 'war_period','pre_corona_vs_post_war','pre_corona_vs_pre_war', 'pre_war_vs_post_war']
for df_type in dfs:
    # Load data
    for drug in drugs:
        for option in options:
            df = pd.read_csv(
                rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\{df_type}\{drug}.csv"
            )

            df = df.sort_values(by=['total_amount','laboratory','role','sold_to_agent',
                    'importation_offence', 'production_offence', 'use_offence', 'self_use',
                    'Court','district', 'Location','Judges_Ethnic', 'Defendants_Ethnic',
                    'marital_status', 'age', 'Date'], 
                ascending=[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
            ).reset_index(drop=True)    

            df = df.reset_index(drop=True)
            pairs = []
            TREATMENT_VALUE = 0
            CONTROL_VALUE = 1
            
            while len(df) > 1:
                # Find first treatment case
                treatment_rows = df[df[option] == TREATMENT_VALUE]
                
                if len(treatment_rows) == 0:
                    break
                
                first_row = treatment_rows.iloc[0]
                
                # Find nearest control (anywhere in the dataframe, not just below)
                candidates = df[df[option] == CONTROL_VALUE]
                
                if len(candidates) == 0:
                    # No controls left
                    break
                
                # Find the closest control by index (nearest in sorted order)
                distances = abs(candidates.index - first_row.name)
                closest_idx = candidates.index[distances.argmin()]
                matched_row = df.loc[closest_idx]
                
                pairs.append({
                    "id_treat": first_row["original_case"],
                    "id_control": matched_row["original_case"]
                })
                
                # Remove both rows
                df = df.drop([first_row.name, closest_idx]).reset_index(drop=True)
            # Save result
            pairs_df = pd.DataFrame(pairs)
            pairs_df.to_csv(f"data/pairs/year/{df_type}/{drug}_{option}.csv", index=False)
            print(f"Created {len(pairs_df)} year-based pairs for {drug} ({df_type}) using {option}")

# build data

## download data

In [ ]:
import pandas as pd
df1 = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_part1.csv")
df2 = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_part2.csv")

# normalize headers (avoid hidden spaces issues)
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

# attach the new column from df2 to df1 by original_case
df_combined = df1.merge(
    df2[["original_case", "has_independent_offenses"]],
    on="original_case",
    how="left"   # keep all rows from df1
)

df_combined.to_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv",
    index=False
)


## orginaize columns

In [ ]:
import pandas as pd
file_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

#in role column remove "" from all values
df['role'] = df['role'].str.replace('"', '')
df['Criminal_record'] = df['Criminal_record'].str.replace('"', '')
df['marital_status'] = df['marital_status'].str.replace('"', '')
df['confession'] = df['confession'].str.replace('"', '')
df['regret'] = df['regret'].str.replace('"', '')
df['responsabillity'] = df['responsabillity'].str.replace('"', '')

df.to_csv(file_path, encoding="utf-8-sig", index=False)


In [ ]:
text = ['לא עבר פלילי','לא עבד עבר פלילי','לבא עבר פלילי','להרשעתו/ה קודמות','לא עלול','#VALUE!','לא ראוי שמחשב את הפסק הדין','- ללא עבר פלילי']
# in Criminal_record columns replace the values in the text list with 'No Criminal Record'
df['Criminal_record'] = df['Criminal_record'].replace(text, 'ללא עבר פלילי')

text = ['אין לי גישה לפסק הדין לפי המספרים שציינת. שלח/י את טקסט פסק הדין או קטע רלוונטי (או ספק קישור/גישה), ואז אענה "כן" או "לא".']
df['self_use'] = df['self_use'].replace(text, 'כן')

df.to_csv(file_path, encoding="utf-8-sig", index=False)

## remove more than 1 defendants

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
print("Initial rows:", len(df))

# remove rows that in age there is the word נאשם more than once
df['accused_count'] = df['role'].apply(lambda x: str(x).count('נאשם'))
df = df[df['accused_count'] <= 1]   
print(len(df))

df['accused_count'] = df['בפועל'].apply(lambda x: str(x).count('נאשם'))
df = df[df['accused_count'] <= 1] 

print("Rows after removing multiple defendants:", len(df))
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")

## extract all amounts and offences

In [ ]:
import pandas as pd
import json
import re

def fix_json_quotes(text):
    """מתקן גרשיים בעייתיים ב-JSON"""
    if pd.isna(text) or text == '':
        return None
    
    text = str(text)
    
    # תיקון גרשיים בין אותיות עבריות
    text = re.sub(r'([א-ת])"([א-ת])', r"\1'\2", text)
    
    # תיקון יחידות מידה
    fixes = {
        'מ"ל': "מ'ל",
        'ש"ח': "ש'ח", 
        'כ"ו': "כ'ו",
        'כמ"ג': "כמ'ג",
        'ג"ר': "ג'ר",
        'ק"ג': "ק'ג",
    }
    
    for old, new in fixes.items():
        text = text.replace(old, new)
    
    # תיקון מבני JSON פגומים
    text = re.sub(r',\s*\[\]\s*\}', r', "sign_b": []}', text)
    text = re.sub(r'"sign_a_personal":\s*\[\],\s*\[\]', r'"sign_a_personal": [], "sign_b": []', text)
    
    return text

def discover_json_fields(df, source_column, verbose=True):

    all_fields = set()
    is_array = False
    has_data = False
    
    # עמודת עזר מתוקנת
    temp_col = f'{source_column}_temp'
    df[temp_col] = df[source_column].apply(fix_json_quotes)
    
    for idx in df.index:
        fixed_val = df.at[idx, temp_col]
        if fixed_val:
            try:
                data = json.loads(fixed_val)
                has_data = True
                
                # אם זה dictionary
                if isinstance(data, dict):
                    all_fields.update(data.keys())
                
                # אם זה list של dictionaries
                elif isinstance(data, list):
                    is_array = True
                    for item in data:
                        if isinstance(item, dict):
                            all_fields.update(item.keys())
            except:
                pass
    
    # מחיקת עמודת העזר
    df = df.drop(temp_col, axis=1)
    
    fields_list = sorted(list(all_fields))
    
    if verbose:
        if not has_data:
            print(f"⚠️  לא נמצא JSON תקין בעמודה '{source_column}'")
        elif not fields_list:
            print(f"⚠️  לא נמצאו שדות JSON בעמודה '{source_column}'")
        else:
            array_note = " (רשימה של אובייקטים)" if is_array else ""
            print(f"🔍 נמצאו {len(fields_list)} שדות בעמודה '{source_column}'{array_note}:")
            for field in fields_list:
                print(f"   • {field}")
    
    return fields_list, is_array

def extract_json_to_columns(df, source_column, prefix='', array_strategy='json', verbose=True):

    
    if source_column not in df.columns:
        print(f"❌ עמודה '{source_column}' לא קיימת")
        return df, {}
    
    stats = {
        'rows_processed': 0,
        'rows_failed': 0,
        'columns_created': [],
        'columns_filled': {},
        'is_array': False
    }
    
    if verbose:
        print("="*60)
        print(f"🔄 מפרק JSON מעמודה: {source_column}")
        print("="*60 + "\n")
    
    # גילוי אוטומטי של כל השדות
    target_columns, is_array = discover_json_fields(df, source_column, verbose=verbose)
    stats['is_array'] = is_array
    
    if not target_columns:
        return df, stats
    
    if verbose and is_array:
        strategy_desc = {
            'json': 'שומר כ-JSON',
            'first': 'לוקח ערך ראשון',
            'concat': 'מאחד עם פסיק'
        }
        print(f"   📌 אסטרטגיה לרשימות: {strategy_desc.get(array_strategy, array_strategy)}\n")
    
    # עמודת עזר מתוקנת
    df[f'{source_column}_fixed'] = df[source_column].apply(fix_json_quotes)
    
    # יצירת עמודות חדשות (רק אם הן לא קיימות)
    for field in target_columns:
        col_name = f"{prefix}{field}" if prefix else field
        if col_name not in df.columns:
            df[col_name] = None
            stats['columns_created'].append(col_name)
    
    # מילוי הנתונים
    for idx in df.index:
        fixed_val = df.at[idx, f'{source_column}_fixed']
        
        if not fixed_val:
            continue
        
        try:
            data = json.loads(fixed_val)
            
            if isinstance(data, dict):
                # טיפול באובייקט רגיל
                for field in target_columns:
                    col_name = f"{prefix}{field}" if prefix else field
                    
                    if field in data:
                        value = data[field]
                        
                        if isinstance(value, (list, dict)):
                            value = json.dumps(value, ensure_ascii=False)
                        
                        current_val = df.at[idx, col_name]
                        if pd.isna(current_val) or current_val == '':
                            df.at[idx, col_name] = value
                            
                            if col_name not in stats['columns_filled']:
                                stats['columns_filled'][col_name] = 0
                            stats['columns_filled'][col_name] += 1
            
            elif isinstance(data, list) and len(data) > 0:
                # טיפול ברשימה של אובייקטים
                for field in target_columns:
                    col_name = f"{prefix}{field}" if prefix else field
                    
                    # איסוף כל הערכים של השדה הזה מהרשימה
                    values = []
                    for item in data:
                        if isinstance(item, dict) and field in item:
                            val = item[field]
                            if val not in ['', None]:  # מתעלם מערכים ריקים
                                values.append(str(val))
                    
                    if values:
                        # החלטה איך לשמור את הערכים
                        if array_strategy == 'first':
                            final_value = values[0]
                        elif array_strategy == 'concat':
                            final_value = ', '.join(values)
                        else:  # 'json' - ברירת מחדל
                            final_value = json.dumps(values, ensure_ascii=False) if len(values) > 1 else values[0]
                        
                        if col_name in df.columns:
                            current_val = df.at[idx, col_name]
                            if pd.isna(current_val) or current_val == '':
                                df.at[idx, col_name] = final_value
                                
                                if col_name not in stats['columns_filled']:
                                    stats['columns_filled'][col_name] = 0
                                stats['columns_filled'][col_name] += 1
            
            stats['rows_processed'] += 1
        
        except Exception as e:
            stats['rows_failed'] += 1
            if verbose:
                print(f"  ⚠️ שגיאה בשורה {idx}: {str(e)[:80]}")
    
    # מחיקת עמודת העזר
    df = df.drop(f'{source_column}_fixed', axis=1)
    
    if verbose:
        print(f"\n📊 סיכום:")
        print(f"   ✅ שורות עובדו: {stats['rows_processed']}")
        print(f"   ❌ שגיאות: {stats['rows_failed']}")
        print(f"   📝 עמודות נוצרו: {len(stats['columns_created'])}")
        if stats['columns_filled']:
            print(f"\n   תאים שמולאו לפי עמודה:")
            for col, count in sorted(stats['columns_filled'].items()):
                print(f"      • {col}: {count}")
    
    return df, stats

def extract_multiple_json_columns(df, source_columns, array_strategy='json', verbose=True):

    # המרה לדיקשנרי אם זה רשימה
    if isinstance(source_columns, list):
        source_columns = {col: {'prefix': '', 'array_strategy': array_strategy} for col in source_columns}
    
    all_stats = {}
    
    for i, (source_col, config) in enumerate(source_columns.items()):
        if isinstance(config, str):
            config = {'prefix': config, 'array_strategy': array_strategy}
        
        prefix = config.get('prefix', '')
        col_array_strategy = config.get('array_strategy', array_strategy)
        
        df, stats = extract_json_to_columns(
            df, 
            source_col, 
            prefix=prefix,
            array_strategy=col_array_strategy,
            verbose=verbose
        )
        
        all_stats[source_col] = stats
        
        if verbose and i < len(source_columns) - 1:
            print("\n" + "="*60 + "\n")
    
    return df, all_stats

# ====================================================================
# שימוש בקוד
# ====================================================================

if __name__ == "__main__":
    # טעינת הקובץ
    file_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv"
    df = pd.read_csv(file_path, encoding="utf-8-sig")
    
    print(f"📂 נטען קובץ עם {len(df)} שורות\n")
    
    # אופציה 1: פשוט - כל העמודות באותה אסטרטגיה
    source_columns = [
        'age',
        'all_drug_offenses',
        'main_sentence',
    ]
    
    
    df, all_stats = extract_multiple_json_columns(df, source_columns, array_strategy='concat', verbose=False)
    
    # # }
    # # df, all_stats = extract_multiple_json_columns(df, source_columns, verbose=True)
    
    # # הצגת דוגמאות
    # print("\n" + "="*60)
    # print("📝 דוגמאות לנתונים שחולצו:")
    # print("="*60)
    
    # for i, idx in enumerate(df.index[:3]):
    #     print(f"\n🔍 שורה {idx}:")
        
    #     # הצגת דוגמאות מעמודות מעניינות
    #     example_cols = ['age_now', 'birth_year', 'section_7', 'בפועל', 'על תנאי', 'קנס']
    #     for col in example_cols:
    #         if col in df.columns and pd.notna(df.at[idx, col]) and df.at[idx, col] != '':
    #             val = str(df.at[idx, col])[:100]
    #             print(f"   • {col}: {val}")
    
    # שמירת הקובץ
    output_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv"
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    
    print(f"\n✅ הקובץ נשמר ב:\n   {output_path}")

## parse the punishment and add metadata

In [ ]:
import pandas as pd
import re
from typing import Dict, Any
import pandas as pd
from datetime import datetime

import pandas as pd
import re

PRIORITY_ORDER = [
    "בפועל",
    "עבודות שירות",
    "על תנאי",
    "שלצ",
    "קנס"
]

def extract_months(text: str) -> float:
    """
    Extract numeric value from strings like:
    '66 חודשי מאסר', '1.17 חודשי מאסר בעבודות שירות'
    """
    if not isinstance(text, str) or not text.strip():
        return 0.0
    match = re.search(r'(\d+(\.\d+)?)', text)
    return float(match.group(1)) if match else 0.0

def build_sentence_columns(df: pd.DataFrame) -> pd.DataFrame:
    sentence_types = []
    sentence_months = []
    punishment_months = []
    is_actual_sentences = []

    for _, row in df.iterrows():
        chosen_type = ""
        chosen_value = ""

        # choose sentence type by priority
        for col in PRIORITY_ORDER:
            if col in df.columns and isinstance(row[col], str) and row[col].strip():
                chosen_type = col
                chosen_value = row[col]
                break

        months = extract_months(chosen_value)

        # is_actual_sentence
        is_actual = chosen_type in ["בפועל", "עבודות שירות"]

        # punishment_months (custodial)
        pm = months if is_actual else 0.0

        sentence_types.append(chosen_type)
        sentence_months.append(months)
        punishment_months.append(pm)
        is_actual_sentences.append(is_actual)


    df["sentence_type"] = sentence_types
    df["sentence_months"] = sentence_months
    df["punishment_months"] = punishment_months
    df['is_actual_sentence'] = is_actual_sentences

    return df


# Helper function to extract number from [480.53-גרם] format
def extract_quantity(val):
    if pd.isna(val) or str(val).strip() == "":
        return None
    match = re.search(r'\[([0-9.]+)-', str(val))
    if match:
        return float(match.group(1))
    return None

# Helper function to convert section column to binary (1/0)
def to_binary(val):
    if pd.isna(val):
        return 0
    val_str = str(val).strip().lower()
    if val_str in ['', 'לא', 'nan']:
        return 0
    return 1
drug_cols = ['lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
             'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
             'mdma', 'cocaine', 'heroin']

location_mapping = {
    "תל אביב-יפו": "Tel Aviv",

    "ראשון לציון": "Center",
    "רמלה": "Center",
    "רחובות": "Center",
    "פתח תקווה": "Center",
    "פתח תקוה": "Center",
    "מרכז": "Center",
    "כפר סבא": "Center",
    "נתניה": "Center",

    "באר שבע": "South",
    "קריית גת": "South",
    "ק' גת": "South",
    "אשקלון": "South",
    "אילת": "South",
    "אשדוד": "South",
    "דימונה": "South",

    "בית שמש": "Jerusalem",
    "ירושלים": "Jerusalem",

    "קי שמונה": "North",  # move to "North" if you meant קריית שמונה
    "ק' שמונה": "North",
    "נצרת": "North",
    "טבריה": "North",
    "עפולה": "North",
    "קצרין": "North",
    "צפת": "North",
    "בית שאן": "North",

    "חיפה": "Haifa",
    "חדרה": "Haifa",
    "עכו": "Haifa",
    "קריות": "Haifa"
}

# Load data
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
print("Initial rows:", len(df))
# Apply parser
df = build_sentence_columns(df)

# Extract just the number from the drug column
# for col in drug_cols:
#     df[col] = df[col].apply(extract_quantity)

# Create new offense columns
df['production_offence'] = df['section_6'].apply(to_binary)      # ייצור סמים
df['use_offence'] = df['section_7'].apply(to_binary)             # שימוש והחזקה
df['importation_offence'] = df['section_13'].apply(to_binary)    # ייבוא/יצוא, מסחר

# Columns to add from verdicts
cols_to_add = ['Date', 'Court', 'Location', 'Judges', 'Prosecutors', 'Defendants']

# Load the verdicts CSV with additional columns
verdicts_df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\initial_data\one_defendant.csv", encoding="utf-8-sig")

# Merge with verdicts to add Date, Court, Location, Judges, Prosecutors, Defendants
df = df.merge(
    verdicts_df[['verdict'] + cols_to_add],
    left_on='original_case',
    right_on='verdict',
    how='left'
)

# Drop duplicate verdict column if exists
if 'verdict' in df.columns and 'original_case' in df.columns:
    df = df.drop(columns=['verdict'])
print(df.columns)
# add year column extracted from Date
# If format is DD/MM/YYYY (Israeli format)
df['year'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce').dt.year

# If format is DD-MM-YYYY
df['year'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce').dt.year

# If format is YYYY-MM-DD
df['year'] = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce').dt.year

# If mixed formats, try dayfirst=True
df['year'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce').dt.year

# Map Location to region
df['district'] = df['Location'].map(location_mapping).fillna('Other')

all_gender = []
verdicts =[]
for row in df['original_case']:
    lst = ['SH-14-09-23152-186', 'ME-13-12-33828-11', 'SH-15-04-21459-877', 'SH-16-08-39727-191', 'ME-12-08-3155-55']
    if row in lst:
        all_gender.append("unknown")
    path = rf'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\CSV_OUTPUT\{row}.csv'
    df_case = pd.read_csv(path, encoding="utf-8-sig")
    for i in df_case['text']:
        if 'נאשמת' in i:
            all_gender.append('female')
            verdicts.append(row)
            break
        elif 'נאשם' in i:
            verdicts.append(row)
            all_gender.append('male')
            break
#print verdict differences
print(set(df['original_case'].tolist()) - set(verdicts))
# add gender column to df
df['gender'] = all_gender 

CORONA = pd.Timestamp("2020-03-11")
WAR = pd.Timestamp("2023-10-07")
df["date_dt"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

df["corona_period"] = df["date_dt"].apply(
    lambda x: 0 if x < CORONA else 1
)
# Create a column for cases that are post-corona but pre-war


df["war_period"] = df["date_dt"].apply(
    lambda x: 0 if x < WAR else 1
)

# pre corona vs post war
def classify_corona_war(date):
    if pd.isna(date):
        return None
    elif date < CORONA:
        return 0
    elif date >= WAR:
        return 1
    else:
        return None  # "between" period

df["pre_corona_vs_post_war"] = df["date_dt"].apply(classify_corona_war)

def classify_corona(date):
    if pd.isna(date):
        return None
    elif date < CORONA:
        return 0
    elif date > CORONA and date < WAR:
        return 1
    else:
        return None
    
df["pre_corona_vs_pre_war"] = df["date_dt"].apply(classify_corona)

def classify_war(date):
    if pd.isna(date):
        return None
    elif date < WAR and date > CORONA:
        return 0
    elif date >= WAR:
        return 1
    else:
        return None
df["pre_war_vs_post_war"] = df["date_dt"].apply(classify_war)

# #remove mdmb column if exists
# if 'mdmb' in df.columns:
#     df = df.drop(columns=['mdmb'])
print("Final rows:", len(df))
# Save
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")


## add punishment range

In [ ]:
import pandas as pd


LOWER_BOUND_DEFAULT = 3


df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\crawler\batch_results_CSV_OUTPUT.csv",
    encoding="utf-8-sig"
)
df1 = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv",
    encoding="utf-8-sig"
)


# Merge
df1 = df1.merge(
    df[['verdict', 'range_low', 'range_high']],
    left_on='original_case',
    right_on='verdict',
    how='left'
)

# Rename to your schema
df1 = df1.rename(columns={
    "range_low": "min_punishment",
    "range_high": "max_punishment"
})
def fill_min_punishment(row):
    if pd.isna(row['min_punishment']) and pd.notna(row['max_punishment']):
        return LOWER_BOUND_DEFAULT
    return row['min_punishment']
df1['min_punishment'] = df1.apply(fill_min_punishment, axis=1)

# Create range_punishment
df1["range_punishment"] = df1["max_punishment"] - df1["min_punishment"]

df1['relative_range_width'] = (
    (df1['max_punishment'] - df1['min_punishment']) /
    df1['max_punishment']
).where(df1['max_punishment'] > 0)

df1.to_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv",
    index=False,
    encoding="utf-8-sig"
)


## fix ages

In [ ]:
import pandas as pd
import re
from datetime import datetime
import math

# -------------------------
# Helpers
# -------------------------
def parse_iso(d):
    if d is None:
        return None

    # אם זה כבר date/datetime
    if isinstance(d, datetime):
        return d.date()

    s = str(d).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None

    # מחרוזות "רשימה ריקה" וכד'
    if s in {"[]", "[ ]", "{}", "{ }"}:
        return None

    # חלץ תאריך ISO מתוך טקסט (גם אם יש שעה)
    m = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", s)
    if not m:
        return None

    try:
        return datetime.strptime(m.group(1), "%Y-%m-%d").date()
    except ValueError:
        return None


def norm_null(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"nan", "none", "null"}:
            return None
        if s in {"[]", "[ ]", "{}", "{ }"}:
            return None
    return x


def compute_age_at_offense(extracted: dict):
    explicit = norm_null(extracted.get("explicit_age_at_offense"))
    if explicit is not None:
        # explicit should be numeric; if it's a string, extract digits
        if isinstance(explicit, str):
            m = re.search(r"\b(\d{1,3})\b", explicit)
            explicit = int(m.group(1)) if m else None
        if explicit is not None:
            return int(explicit)

    offense_date = norm_null(extracted.get("offense_date"))
    offense_dates = extracted.get("offense_dates") or []
    sentencing_date = norm_null(extracted.get("sentencing_date"))

    if (offense_date is None) and offense_dates:
        if sentencing_date:
            sd = parse_iso(sentencing_date)
            ods = [parse_iso(x) for x in offense_dates if x]
            ods = [x for x in ods if x]
            if ods:
                offense_date = min(ods, key=lambda x: abs((sd - x).days))
        if offense_date is None:
            offense_date = offense_dates[0]

    od = parse_iso(offense_date) if offense_date else None

    bd = parse_iso(norm_null(extracted.get("birth_date")))
    if bd and od:
        return math.floor((od - bd).days / 365.25)

    age_now = norm_null(extracted.get("age_now"))
    if isinstance(age_now, str):
        m = re.search(r"\b(\d{1,3})\b", age_now)
        age_now = int(m.group(1)) if m else None

    ref_date_str = norm_null(extracted.get("age_reference_date")) or norm_null(extracted.get("sentencing_date"))
    ref = parse_iso(ref_date_str) if ref_date_str else None

    if age_now is not None and ref and od:
        delta_years = ((ref - od).days / 365.25)
        #round down or up to nearest integer
        
        delta_years = math.floor(((ref - od).days / 365.25))
        return int(age_now) - delta_years
    
    # --- fallback: birth_year + offense_date ---
    birth_year = norm_null(extracted.get("birth_year"))
    if birth_year is not None and od:
        try:
            by = int(birth_year)
            return od.year - by
        except Exception:
            pass

    # --- fallback: birth_year + sentencing_date ---
    if birth_year is not None and sentencing_date:
        sd = parse_iso(sentencing_date)
        if sd:
            try:
                by = int(birth_year)
                return sd.year - by
            except Exception:
                pass
    if age_now is not None:
        return int(age_now)
    return "לא ידוע"


OFFENSE_TRIGGERS = ["בעת ביצוע", "ביום האירוע", "במועד ביצוע", "בזמן ביצוע", "במועד האירוע", "במועד הרלוונטי"]

def sanitize_explicit_age(d):
    """
    If explicit_age_at_offense is a string like 'הנאשם בן 43',
    keep it ONLY if the same string contains an offense-time trigger.
    Otherwise set to None (it's age_now, not explicit at offense).
    """
    x = d.get("explicit_age_at_offense")
    if isinstance(x, str):
        m = re.search(r"\b(\d{1,3})\b", x)
        num = int(m.group(1)) if m else None

        if any(t in x for t in OFFENSE_TRIGGERS):
            d["explicit_age_at_offense"] = num
        else:
            d["explicit_age_at_offense"] = None
            # optional: if age_now missing, use the number as age_now
            if d.get("age_now") in (None, "", "nan") and num is not None:
                d["age_now"] = num
    return d

def parse_offense_dates_cell(val):
    """
    offense_dates might be:
    - NaN / empty
    - a single ISO string
    - a stringified list like "['2021-06-05', '2021-06-10']"
    - a JSON list string like '["2021-06-05"]'
    Normalize to list[str].
    """
    if val is None:
        return []
    s = str(val).strip()
    if s == "" or s.lower() == "nan":
        return []

    # already looks like a list
    if s.startswith("[") and s.endswith("]"):
        # try JSON style first
        try:
            import json
            out = json.loads(s)
            if isinstance(out, list):
                return [str(x).strip().strip('"').strip("'") for x in out if str(x).strip()]
        except Exception:
            pass
        # fallback: extract ISO dates by regex
        return re.findall(r"\d{4}-\d{2}-\d{2}", s)

    # single date
    m = re.search(r"\d{4}-\d{2}-\d{2}", s)
    return [m.group(0)] if m else []

# -------------------------
# Load
# -------------------------
df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv",
    encoding="utf-8-sig"
)

# -------------------------
# Compute new column
# -------------------------
AGE_FIELDS = [
    "explicit_age_at_offense",
    "age_now",
    "age_reference_date",
    "offense_date",
    "offense_dates",
    "sentencing_date",
    "birth_date",
    "birth_year",
]

def compute_age_from_columns(row):
    extracted = {k: norm_null(row.get(k, None)) for k in AGE_FIELDS}

    extracted["offense_dates"] = parse_offense_dates_cell(extracted.get("offense_dates"))
    extracted = sanitize_explicit_age(extracted)

    return compute_age_at_offense(extracted)


# Create a NEW column (don’t overwrite your extracted columns)
df["age"] = df.apply(compute_age_from_columns, axis=1)
# IN ROW WHERE original_case is SH-14-09-23152-186, set age to 43 (known from external source)
df.loc[df['original_case'] == 'ME-22-08-10639-776', 'age'] = 19
df.loc[df['original_case'] == 'SH-18-09-8588-179', 'age'] = 18
df.loc[df['original_case'] == 'SH-20-06-70552-55', 'age'] = 20
df.loc[df['original_case'] == 'SH-21-01-11879-22', 'age'] = 18
df.loc[df['original_case'] == 'SH-21-02-57960-11', 'age'] = 17
df.loc[df['original_case'] == 'SH-19-03-44336-44', 'age'] = 33
df.loc[df['original_case'] == 'ME-21-05-16015-526', 'age'] = 20

#where age is smaller than 15 change to לא ידוע
#change age to numeric, if there are non-numeric values (like "לא ידוע"), set them to NaN
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df.loc[df['age'] < 15, 'age'] = None

# change from nan to "לא ידוע"
df['age'] = df['age'].fillna("לא ידוע")

TREATMENT_VALUE = "Young"
CONTROL_VALUE = "Older"
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age_binary"] = df["age"].apply(lambda x: TREATMENT_VALUE if x <= 28 else CONTROL_VALUE)
#make from nan to "לא ידוע" in age_binary
df['age'] = df['age'].fillna("לא ידוע")
# -------------------------
# Save (prefer saving to a new file first)
# -------------------------
out_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)


## replace to 1/0

In [ ]:
binary_cols = ['confession', 'responsabillity', 'regret','self_use', 'laboratory','sold_to_agent']  # הוסף עוד אם צריך
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
# df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\drug_features_with_punishmentsRanges_cleanedAge.csv", encoding="utf-8-sig")

# print age nan count and not nan count
print(f"\nAge missing count: {df['age'].isna().sum()}")
print(f"Age not missing count: {df['age'].notna().sum()}")
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].replace({'כן': 1, 'לא': 0})
        df[col] = pd.to_numeric(df[col], errors='coerce')


# שמור
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")


## add ethnicity

In [ ]:

import pandas as pd
from crawler.name_classifier import NameClassifier, process_dataframe

def ethnic(INPUT_FILE, OUTPUT_FILE):
    # ============================================
    # הגדרות - שנה את הנתיבים לפי הצורך
    # ============================================
    
    # קובץ קלט
    INPUT_FILE = INPUT_FILE
    
    # קובץ פלט
    OUTPUT_FILE = OUTPUT_FILE
    
    # נתיבים לקבצי השמות
    FIRST_NAMES_PATH = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\csv_for_race\first_names.xlsx'
    JEWISH_LAST_PATH = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\csv_for_race\jewish_last_names.xlsx'
    MUSLIM_LAST_PATH = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\csv_for_race\muslims_last_names.xlsx'
    CHRISTIAN_LAST_PATH = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\csv_for_race\christians_last_names.xlsx'
    DRUZE_LAST_PATH = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\csv_for_race\druze_last_names.xlsx'
    
    # שמות עמודות
    DEFENDANTS_COL = 'Defendants'
    JUDGES_COL = 'Judges'
    
    # פרמטרים
    LAST_NAME_WEIGHT = 1.5  # משקל לשם משפחה (כפול משם פרטי)
    USE_FUZZY = True        # האם להשתמש ב-Levenshtein matching
    
    # ============================================
    # הרצה
    # ============================================
    
    print("="*60)
    print("Name Classification Script")
    print("="*60)
    
    # Initialize classifier
    print("\nInitializing classifier...")
    classifier = NameClassifier(
        first_names_path=FIRST_NAMES_PATH,
        last_names_paths={
            'Jewish': JEWISH_LAST_PATH,
            'Muslim': MUSLIM_LAST_PATH,
            'Christian': CHRISTIAN_LAST_PATH,
            'Druze': DRUZE_LAST_PATH
        },
        last_name_weight=LAST_NAME_WEIGHT,
        use_fuzzy=USE_FUZZY
    )
    
    # Load data
    print(f"\nLoading {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df)} rows")
    
    # Process defendants if column exists
    if DEFENDANTS_COL in df.columns:
        print(f"\nProcessing {DEFENDANTS_COL}...")
        df = process_dataframe(df, classifier, DEFENDANTS_COL, is_multiple=False)
        print(f"Distribution:")
        print(df[f'{DEFENDANTS_COL}_Ethnic'].value_counts())
    else:
        print(f"Warning: Column '{DEFENDANTS_COL}' not found")
    
    # Process judges if column exists
    if JUDGES_COL in df.columns:
        print(f"\nProcessing {JUDGES_COL}...")
        df = process_dataframe(df, classifier, JUDGES_COL, is_multiple=True)
        print(f"Distribution:")
        print(df[f'{JUDGES_COL}_Ethnic'].value_counts())
    else:
        print(f"Warning: Column '{JUDGES_COL}' not found")
    
    # Save results
    print(f"\nSaving to {OUTPUT_FILE}...")
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print("Done!")
    
    # Show sample
    print("\n" + "="*60)
    print("Sample results (first 10 rows):")
    print("="*60)
    cols_to_show = [DEFENDANTS_COL] if DEFENDANTS_COL in df.columns else []
    cols_to_show += [c for c in df.columns if 'Ethnic' in c or 'Confidence' in c]
    print(df[cols_to_show].head(10).to_string())


In [ ]:
OUTPUT_FILE = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv'
INPUT_FILE = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv'
ethnic(INPUT_FILE,OUTPUT_FILE)

## normalize amounts 

In [ ]:
import ast
import re
import numpy as np
import pandas as pd

# =========================
# CONSTANTS
# =========================
SIGNS = ['sign_a_non_personal', 'sign_a_personal', 'sign_b']

drug_cols = [
    'lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
    'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
    'mdma', 'cocaine', 'heroin'
]

# =========================
# 1) DEDUPLICATION BY SIGN PRIORITY
# sign_b > sign_a_personal > sign_a_non_personal
# =========================
def dedupe_drug_cell(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return cell

    if isinstance(cell, str):
        try:
            cell = ast.literal_eval(cell)
        except Exception:
            return cell

    if not isinstance(cell, dict):
        return cell

    def canon(x):
        if x is None:
            return ""
        s = str(x)
        s = re.sub(r"\s+", "", s)
        s = s.replace("״", '"').replace("׳", "'")
        s = re.sub(r"(?<=\d),(?=\d)", ".", s)  # 0,5 -> 0.5
        return s

    non_p = cell.get('sign_a_non_personal', []) or []
    pers  = cell.get('sign_a_personal', []) or []
    b     = cell.get('sign_b', []) or []

    non_p_map = {canon(x): x for x in non_p}
    pers_map  = {canon(x): x for x in pers}
    b_map     = {canon(x): x for x in b}

    # remove duplicates from non_personal if exist in higher priority
    for k in list(non_p_map.keys()):
        if k in pers_map or k in b_map:
            non_p_map.pop(k, None)

    cell['sign_a_non_personal'] = list(non_p_map.values())
    cell['sign_a_personal'] = list(pers_map.values())
    cell['sign_b'] = list(b_map.values())

    return cell

def remove_similar_quantities(quantity_list, tolerance_percent=0.5):
    """
    Remove quantities that are very close to each other (likely duplicates with different precision)
    ומ
    Args:
        quantity_list: list of quantity strings like ['8.717-ק"ג', '8.7-ק"ג']
        tolerance_percent: if two quantities differ by less than this %, consider them duplicates
    
    Returns:
        filtered list with similar quantities removed (keeps the one with more precision)
    """
    if not quantity_list or len(quantity_list) <= 1:
        return quantity_list
    
    # Parse each quantity into (amount, unit, original_string, precision)
    parsed = []
    for q_str in quantity_list:
        if not q_str:
            continue
        
        # Clean up
        clean_str = q_str.replace('\\"', '"').replace("\\'", "'")
        clean_str = clean_str.replace('״', '"').replace('׳', "'")
        
        # Extract amount and unit
        match = re.match(r'^\[?([0-9.,]+)\s*-\s*([^\]]+)\]?$', clean_str.strip())
        if match:
            amount_str = match.group(1).replace(',', '.')
            unit = match.group(2).strip()
            try:
                amount = float(amount_str)
                # Count decimal places for precision
                precision = len(amount_str.split('.')[-1]) if '.' in amount_str else 0
                parsed.append({
                    'amount': amount,
                    'unit': unit,
                    'original': q_str,
                    'precision': precision
                })
            except ValueError:
                parsed.append({'original': q_str})  # Keep unparseable as-is
    
    if not parsed:
        return quantity_list
    
    # Group by unit
    by_unit = {}
    unparseable = []
    
    for item in parsed:
        if 'amount' in item:
            unit = item['unit']
            if unit not in by_unit:
                by_unit[unit] = []
            by_unit[unit].append(item)
        else:
            unparseable.append(item['original'])
    
    # For each unit, remove similar quantities
    result = []
    for unit, items in by_unit.items():
        if len(items) == 1:
            result.append(items[0]['original'])
            continue
        
        # Sort by amount
        items_sorted = sorted(items, key=lambda x: x['amount'])
        
        # Mark items to keep
        keep = [True] * len(items_sorted)
        
        for i in range(len(items_sorted)):
            if not keep[i]:
                continue
            
            for j in range(i + 1, len(items_sorted)):
                if not keep[j]:
                    continue
                
                amt_i = items_sorted[i]['amount']
                amt_j = items_sorted[j]['amount']
                
                # Calculate percentage difference
                avg = (amt_i + amt_j) / 2
                diff_percent = abs(amt_i - amt_j) / avg * 100
                
                if diff_percent <= tolerance_percent:
                    # Keep the one with more precision (more decimal places)
                    if items_sorted[i]['precision'] >= items_sorted[j]['precision']:
                        keep[j] = False
                    else:
                        keep[i] = False
                        break
        
        # Add kept items
        for i, item in enumerate(items_sorted):
            if keep[i]:
                result.append(item['original'])
    
    result.extend(unparseable)
    return result

# =========================
# 2) EXTRACT ALL QUANTITIES (AFTER DEDUPE)
# =========================
def extract_all_quantities(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return []

    if isinstance(cell, str):
        try:
            cell = ast.literal_eval(cell)
        except Exception:
            return []

    if not isinstance(cell, dict):
        return []

    quantities = []
    for sign in SIGNS:
        vals = cell.get(sign, [])
        if isinstance(vals, list):
            quantities.extend(vals)

    return quantities

# =========================
# 3) NORMALIZE QUANTITIES (SUM)
# =========================
def normalize_drug_quantity(quantity_str):
    if pd.isna(quantity_str) or not str(quantity_str).strip():
        return ""

    quantity_str = str(quantity_str).strip()
    total_grams = 0.0
    total_tablets = 0.0
    total_plants = 0.0
    has_valid = False

    # Clean up escaped quotes first
    quantity_str = quantity_str.replace('\\"', '"').replace("\\'", "'")
    quantity_str = quantity_str.replace('״', '"').replace('׳', "'")

    bracket_content = re.findall(r'\[(.*?)\]', quantity_str)
    all_pairs = []

    for content in bracket_content:
        parts = content.split(',')
        for part in parts:
            part = part.strip()
            match = re.match(r'^([0-9.,]+)\s*-\s*(.+)$', part)
            if match:
                all_pairs.append((match.group(1), match.group(2)))

    if not all_pairs:
        pattern = r'\[([0-9.,]+)-([^\]]+)\]'
        all_pairs = re.findall(pattern, quantity_str)
    
    # If still no pairs found, try to parse without brackets
    if not all_pairs:
        # Handle formats like: 27.5-ק"ג or [27.5-ק"ג]
        pattern = r'([0-9.,]+)\s*-\s*([^\[\],]+)'
        all_pairs = re.findall(pattern, quantity_str)

    for amount_str, unit in all_pairs:
        unit = unit.strip()
        amount_str = amount_str.replace(',', '.').strip()

        try:
            amount = float(amount_str)
            has_valid = True
        except ValueError:
            continue

        if unit in ['ק"ג', 'קילוגרם', 'קילו', 'kg', 'קג']:
            total_grams += amount * 1000
        elif unit in ['גרם', 'gram', 'grams', 'gr', 'g']:
            total_grams += amount
        elif unit in ['מ"ג', 'מיליגרם', 'mg']:
            total_grams += amount / 1000
        elif unit in ['טבליות', 'כדורים', 'יחידות', 'tablets', 'pills', 'בולים', 'כדור', 'טבליה']:
            total_tablets += amount
        elif unit in ['שתילים', 'שתיל', 'plants', 'צמחים']:
            total_plants += amount

    if not has_valid:
        return ""

    parts = []
    if total_grams > 0:
        parts.append(f"[{int(total_grams) if total_grams.is_integer() else round(total_grams, 2)}-גרם]")
    if total_tablets > 0:
        parts.append(f"[{int(total_tablets)}-טבליות]")
    if total_plants > 0:
        parts.append(f"[{int(total_plants)}-שתילים]")

    return ", ".join(parts) if parts else ""

def normalize_from_list(quantity_list):
    if quantity_list is None or not isinstance(quantity_list, list) or len(quantity_list) == 0:
        return ""
    
    # Remove similar quantities first
    filtered_list = remove_similar_quantities(quantity_list, tolerance_percent=0.01)
    
    joined = "[" + ", ".join(filtered_list) + "]"
    return normalize_drug_quantity(joined)

# =========================
# 4) EXTRACT NUMERIC VALUE
# =========================
def extract_numeric_value(normalized_str):
    if pd.isna(normalized_str) or not str(normalized_str).strip():
        return np.nan
    match = re.search(r'\[([0-9.]+)-', str(normalized_str))
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return np.nan
    return np.nan



In [ ]:
# =========================
# 5) PIPELINE EXECUTION
# =========================
df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv",
    encoding="utf-8-sig"
)

for col in drug_cols:
    # 1) remove duplicates by sign priority
    df[col] = df[col].apply(dedupe_drug_cell)

    # 2) flatten quantities
    df[col] = df[col].apply(extract_all_quantities)

    # 3) normalize & sum
    df[col] = df[col].apply(normalize_from_list)

    # 4) extract numeric value
    df[col] = df[col].apply(extract_numeric_value)


df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")

## remove female

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
# remove rows where in gender column the value is female 
print(len(df))
df = df[df['gender'] != 'female']
print(len(df))
# save the result to the same file
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")

## remove rows with no amounts

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
print("Initial rows:", len(df))
#remove rows where ther no drug quantities at all in the drug columns
drug_cols = ['lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
             'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
             'mdma', 'cocaine', 'heroin']
df['has_drug_quantity'] = df[drug_cols].apply(lambda row: any(pd.notna(val) and str(val).strip() != "" for val in row), axis=1)
df_with_quantities = df[df['has_drug_quantity']]
#remove the helper column
df_with_quantities = df_with_quantities.drop(columns=['has_drug_quantity'])
print(f"Rows with drug quantities: {len(df_with_quantities)}")
df_with_quantities.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")


## remove rows with multiply drugs

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
drug_cols = ['lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
             'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
             'mdma', 'cocaine', 'heroin']

# remove rows with multiply drugs (except cannabis + cannabis_plants can be together)
print("Initial rows:", len(df))

# Count number of drugs, treating cannabis and cannabis_plants as one
def count_drugs(row):
    # Count all drugs
    drug_count = sum(pd.notna(val) and str(val).strip() != "" for col, val in zip(drug_cols, row))
    
    # If both cannabis and cannabis_plants are present, reduce count by 1
    has_cannabis = pd.notna(row['cannabis']) and str(row['cannabis']).strip() != ""
    has_cannabis_plants = pd.notna(row['cannabis_plants']) and str(row['cannabis_plants']).strip() != ""
    
    if has_cannabis and has_cannabis_plants:
        drug_count -= 1
    
    return drug_count

df['num_drugs'] = df[drug_cols].apply(count_drugs, axis=1)
df_single_drug = df[df['num_drugs'] == 1].copy()
print("Rows after removing multiple drugs (cannabis + cannabis_plants allowed):", len(df_single_drug))
df_single_drug = df_single_drug.drop(columns=['num_drugs'])
df_single_drug.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")
print("✅ Saved!")

## remove files that not only drugs

In [ ]:

df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
# remove rows where in has_independent_offenses is כן
print("Initial rows:", len(df))
df_no_independent = df[df['has_independent_offenses'] != 'כן']
print("Rows after removing independent offenses:", len(df_no_independent))
df_no_independent.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")
print("✅ Saved!")

## remove unnecessery columns

In [ ]:

df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
columns_to_keep = ['original_case','lsd', 'methamphetamine', 'ayahuasca','cathinone', 'ketamine', 'hashish', 'methylmethcathinone', 'cannabis',
       'cannabis_plants', 'mdma', 'cocaine', 'heroin', 'confession','Criminal_record', 'laboratory', 'marital_status', 'regret','responsabillity',
        'role', 'self_use', 'sold_to_agent', 'age', 'punishment_months', 'is_actual_sentence','production_offence', 'use_offence', 'importation_offence',
        'Date','Court', 'Location', 'Judges', 'Defendants', 'year', 'district', 'gender', 'date_dt', 'corona_period', 'war_period','pre_corona_vs_post_war',
        'pre_corona_vs_pre_war', 'pre_war_vs_post_war','min_punishment', 'max_punishment', 'range_punishment','relative_range_width', 'Defendants_Ethnic', 'Defendants_Nationality',
        'Defendants_Confidence', 'Judges_Ethnic', 'Judges_Nationality', 'Judges_Confidence']
df = df[columns_to_keep]

# orginize the columns
columns_to_keep = [
    "original_case",

    # Case metadata
    "Date", "date_dt", "year", "Court", "district", "Location",
    "Judges", "Defendants", "gender",

    # Drug indicators
    "lsd", "methamphetamine", "ayahuasca", "cathinone", "ketamine",
    "hashish", "methylmethcathinone", "cannabis", "cannabis_plants",
    "mdma", "cocaine", "heroin",

    # Offense flags
    "production_offence", "use_offence", "importation_offence",

    # Defendant/case features
    "age", "marital_status", "confession", "responsabillity", "regret",
    "Criminal_record", "laboratory", "role", "self_use", "sold_to_agent",

    # Sentence/punishment
    "is_actual_sentence", "punishment_months", "min_punishment",
    "max_punishment", "range_punishment", "relative_range_width",

    # Time periods
     'corona_period', 'war_period','pre_corona_vs_post_war',
        'pre_corona_vs_pre_war', 'pre_war_vs_post_war',

    # Ethnicity/nationality inference
    "Defendants_Ethnic", "Defendants_Nationality", "Defendants_Confidence",
    "Judges_Ethnic", "Judges_Nationality", "Judges_Confidence",
]

df = df.reindex(columns=columns_to_keep)
print(len(df))
# df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\data_before_punishment.csv", index=False, encoding="utf-8-sig")

## remove rows with no punishment info

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\data_before_punishment.csv", encoding="utf-8-sig")

#remove rows where both range_low or range_high are na
print("Initial rows:", len(df))
# print the rows where  min_punishment and  max_punishment is na
print(
    "Rows with missing min_punishment and max_punishment:\n",
    len(df[df["max_punishment"].isna() & df["min_punishment"].isna()])
)
print("Rows with missing min_punishment:\n", len(df[df['min_punishment'].isna()]))
print("Rows with missing max_punishment:\n", len(df[df['max_punishment'].isna()]))
df = df[~((df['max_punishment'].isna()) & (df['min_punishment'].isna()) | df['punishment_months'].isna())]
print("Rows after removing missing punishments:", len(df))
df.to_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", index=False, encoding="utf-8-sig")
print("✅ Saved!")


## split to cannabis, cocaine, heroin and hashish

In [ ]:
import pandas as pd
import re

# Load the normalized drug features
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv", encoding="utf-8-sig")
print(len(df))

drug_cols = ['lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
             'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
             'mdma', 'cocaine', 'heroin']

# Define cannabis group - these can appear together
cannabis_group = ['cannabis', 'cannabis_plants']

# Helper function to check if value is filled
def is_filled(val):
    return pd.notna(val) and str(val).strip() != ""


# Target drugs for separate CSV
target_drugs = ['cannabis', 'cocaine', 'hashish', 'heroin', 'mdma', 'methamphetamine', 'lsd', 'ketamine', 'ayahuasca', 'cathinone', 'methylmethcathinone']


for target in target_drugs:
    # Start with rows where this drug is filled
    mask = df[target].apply(is_filled)
    
    # Exclude rows where any other drug is filled
    for other_col in drug_cols:
        if other_col != target:
            # Skip if both are in cannabis group
            if target in cannabis_group and other_col in cannabis_group:
                continue
            mask = mask & ~df[other_col].apply(is_filled)
    
    # Filter dataframe
    df_only = df[mask].copy()

    # Rename target column to total_amount
    df_only = df_only.rename(columns={target: 'total_amount'})

    
    # Remove all other drug columns
    cols_to_drop = [col for col in drug_cols if col != target]
    df_only = df_only.drop(columns=cols_to_drop, errors='ignore')

    # Save to CSV
    output_path = rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute\{target}.csv"
    df_only.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ {target}: {len(df_only)} שורות, {len(df_only.columns)} עמודות")
    print()

    # remove rows where age is לא ידוע and remove rows where marital_status is לא ידוע
    df_only = df_only[~((df_only['age'] == 'לא ידוע') | (df_only['marital_status'] == 'לא ידוע'))]
    # Save to CSV
    output_path = rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\no_impute\{target}.csv"
    df_only.to_csv(output_path, index=False, encoding="utf-8-sig")
    
    print(f"✅ {target}: {len(df_only)} שורות, {len(df_only.columns)} עמודות")
    print()

## imputed missing ages and status

In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# Ordinal encoding for Criminal_record (ordered: no record < light < heavy)
CRIMINAL_RECORD_ORDER = {
    'ללא עבר פלילי': 0,
    'עבר פלילי קל':  1,
    'עבר פלילי כבד': 2,
}
CRIMINAL_RECORD_DECODE = {v: k for k, v in CRIMINAL_RECORD_ORDER.items()}

drugs = ['cannabis', 'cocaine', 'hashish']
for drug in drugs:
    df = pd.read_csv(
        rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute\{drug}.csv",
        encoding="utf-8-sig"
    )

    # --- נקה age ---
    def clean_age(val):
        if pd.isna(val):
            return np.nan
        val_str = str(val).strip()
        if val_str == 'לא ידוע' or val_str == '':
            return np.nan
        try:
            return float(val_str)
        except:
            return np.nan

    df['age'] = df['age'].apply(clean_age)

    # --- נקה marital_status והמר לקודים ---
    df['marital_status'] = df['marital_status'].replace('לא ידוע', np.nan)

    status_categories = df['marital_status'].dropna().unique().tolist()
    status_to_code = {status: i for i, status in enumerate(status_categories)}
    code_to_status = {i: status for status, i in status_to_code.items()}
    df['marital_status_code'] = df['marital_status'].map(status_to_code)

    print(f"Marital status mapping: {status_to_code}")

    # --- נקה Criminal_record והמר לקודים אורדינליים ---
    df['Criminal_record'] = df['Criminal_record'].replace('לא ידוע', np.nan)
    df['criminal_record_code'] = df['Criminal_record'].map(CRIMINAL_RECORD_ORDER)

    print(f"Criminal record missing before imputation: {df['Criminal_record'].isna().sum()}")

    # --- הכן משתנים ---
    cols_to_impute = ['age', 'marital_status_code', 'criminal_record_code']

    # עמודות עזר (מספריים)
    helper_numeric = ['confession', 'responsabillity', 'regret', 'self_use', 'production_offence',
                    'use_offence', 'importation_offence', 'laboratory', 'sold_to_agent', 'year', 'total_amount']
    helper_numeric = [c for c in helper_numeric if c in df.columns]
    for col in helper_numeric:
        if col in df.columns:
            df[col] = df[col].replace({'כן': 1, 'לא': 0})
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # עמודות עזר (קטגוריאליות)
    helper_categorical = ['district', 'court', 'ethnic', 'judge_ethnic']
    helper_categorical = [c for c in helper_categorical if c in df.columns]

    # One-hot encoding לקטגוריאליות
    df_encoded = pd.get_dummies(df[helper_categorical], dummy_na=False)

    # חבר הכל
    df_for_imputation = pd.concat([
        df[cols_to_impute],
        df[helper_numeric],
        df_encoded
    ], axis=1)

    # שמור מקוריים (חוץ מהעמודות שרוצים לחזות)
    original_values = {col: df_for_imputation[col].copy()
                    for col in df_for_imputation.columns
                    if col not in cols_to_impute}

    # --- MICE עם Random Forest ---
    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=42),
        max_iter=10,
        random_state=42
    )
    df_imputed = pd.DataFrame(
        imputer.fit_transform(df_for_imputation),
        columns=df_for_imputation.columns
    )

    # --- החזר ערכים ---
    # age
    df['age'] = df_imputed['age'].round().astype(int)

    # marital_status - עגל והמר בחזרה
    df['marital_status_code'] = df_imputed['marital_status_code'].round().astype(int)
    df['marital_status_code'] = df['marital_status_code'].clip(0, len(status_categories) - 1)
    df['marital_status'] = df['marital_status_code'].map(code_to_status)
    df = df.drop(columns=['marital_status_code'])

    # Criminal_record - עגל והמר בחזרה
    criminal_codes = df_imputed['criminal_record_code'].round().astype(int).clip(0, 2)
    df['Criminal_record'] = criminal_codes.map(CRIMINAL_RECORD_DECODE)
    df = df.drop(columns=['criminal_record_code'], errors='ignore')

    # החזר ערכים מקוריים לעמודות העזר
    for col in original_values:
        if col in df.columns:
            df[col] = original_values[col]

    # --- בדיקה ---
    print(f"\n✅ {drug} after imputation:")
    print(f"  Age missing:             {df['age'].isna().sum()}")
    print(f"  Marital status missing:  {df['marital_status'].isna().sum()}")
    print(f"  Criminal_record missing: {df['Criminal_record'].isna().sum()}")
    print(f"  Criminal_record dist:\n{df['Criminal_record'].value_counts().to_string()}")

    # שמור
    df.to_csv(
        rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute\{drug}.csv",
        index=False, encoding="utf-8-sig"
    )


# change  in punishment range into 0/6

In [ ]:
cannabis = pd.read_csv(
rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute\cannabis.csv",
encoding="utf-8-sig"
)

cocaine = pd.read_csv(
rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute\cocaine.csv",
encoding="utf-8-sig"
)  

hashish = pd.read_csv(
rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute\hashish.csv",
encoding="utf-8-sig"
)  
for lower in LOWER_BOUND_DEFAULT:
    cannabis.to_csv(rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute_{lower}\cannabis.csv", index=False, encoding="utf-8-sig")
    cocaine.to_csv(rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute_{lower}\cocaine.csv", index=False, encoding="utf-8-sig")
    hashish.to_csv(rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute_{lower}\hashish.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

LOWER_BOUND_DEFAULT = [0,6]

df = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\crawler\batch_results_CSV_OUTPUT.csv",
    encoding="utf-8-sig"
)

df2 = pd.read_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\features_extractions.csv",
    encoding="utf-8-sig"
)

dfs = ['cannabis', 'cocaine','hashish']
for lower in LOWER_BOUND_DEFAULT:
    for f in dfs:
        df1 = pd.read_csv(
        rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute_{lower}\{f}.csv",
        encoding="utf-8-sig"
        )


        # Merge
        df1 = df1.merge(
            df[['verdict', 'range_low', 'range_high']],
            left_on='original_case',
            right_on='verdict',
            how='left'
        )
        print(len(df1))
        # find rows in df1 where pd.isna(row['min_punishment']) and pd.notna(row['max_punishment']) and verdict is in df2['verdict']
        missing_min = df1[(df1['range_low'].isna()) & (df1['range_high'].notna()) & (df1['min_punishment'] == 3)]
        print(len(missing_min))
    # FOR missing_min CHANGE IN df2 THE MIN INTO LOWER_BOUND_DEFAULT AND SAVE CSV
        df1.loc[df1['original_case'].isin(missing_min['original_case']), 'min_punishment'] = lower
        df1.to_csv(
            rf"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data\impute_{lower}\{f}.csv",
            index=False, encoding="utf-8-sig"
        )

# SM sensitivity data
Data variants for the robustness checks in Section 3.3. These cells only write to `data/no_impute` and `data/unfiltered`; `data/impute` is not modified.

## no_impute – listwise deletion (Section 3.3.2a)
Start from the primary `impute` files, restore the three imputed columns to their original values, and drop every case where any of them is unknown. All other columns stay identical to the primary analysis.

In [ ]:
import pandas as pd

BASE = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\data"
UNKNOWN = "לא ידוע"
PERSONAL_COLS = ["age", "marital_status", "Criminal_record"]
DRUGS = ["cannabis", "cocaine", "hashish"]

# Original (pre-imputation) personal values, by case
features = pd.read_csv(rf"{BASE}\features_extractions.csv", encoding="utf-8-sig")
original_personal = features.set_index("original_case")[PERSONAL_COLS]

def revert_imputation(df):
    df = df.copy()
    df[PERSONAL_COLS] = original_personal.loc[df["original_case"]].values
    return df

for drug in DRUGS:
    df = revert_imputation(pd.read_csv(rf"{BASE}\impute\{drug}.csv", encoding="utf-8-sig"))
    known = (df[PERSONAL_COLS].astype(str) != UNKNOWN).all(axis=1)
    df[known].to_csv(rf"{BASE}\no_impute\{drug}.csv", index=False, encoding="utf-8-sig")
    print(f"✅ {drug}: kept {known.sum()} of {len(df)} rows")
    print(df[known]["ethnic"].value_counts().to_string(), "\n")

## unfiltered – cases without a declared sentencing range (Section 3.3.3)
The 1,755-case sample before excluding verdicts that lack a sentencing range (`data_before_punishment.csv`). Cases in the final sample keep their existing ethnicity; the excluded cases are classified with the same Shayo & Zussman + BISG procedure (`crawler/ethnic.py`). Location priors come from all classified cases in the unfiltered sample. Imputation is the same as the primary `impute` data. Only `is_actual_sentence` is analysed on this sample.

In [ ]:
import sys
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

sys.path.insert(0, r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\crawler")
from ethnic import ShayoZussmanClassifier, process_dataframe

NAMES_DIR = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\csv_for_race"
classifier = ShayoZussmanClassifier(
    first_names_path=rf"{NAMES_DIR}\first_names.xlsx",
    last_names_paths={
        'Jewish': rf"{NAMES_DIR}\jewish_last_names.xlsx",
        'Muslim': rf"{NAMES_DIR}\muslims_last_names.xlsx",
        'Christian': rf"{NAMES_DIR}\christians_last_names.xlsx",
        'Druze': rf"{NAMES_DIR}\druze_last_names.xlsx",
    }
)

def classify_ethnicity(df, name_col, output_col, reference):
    # Step 1: Shayo & Zussman
    df = process_dataframe(df.copy(), classifier, name_col, output_prefix=output_col)
    # Step 2: BISG fallback with court-location priors
    classified = pd.concat([reference[["Location", output_col]], df[["Location", output_col]]])
    classified = classified[classified[output_col].isin(["Arab", "Jewish"])]
    priors = classified.groupby("Location")[output_col].apply(lambda s: (s == "Arab").mean()).to_dict()
    overall_prior = (classified[output_col] == "Arab").mean()
    for idx in df.index[df[output_col] == "Unknown"]:
        p_arab_loc = priors.get(df.at[idx, "Location"], overall_prior)
        result, _ = classifier.bisg_classify_name(df.at[idx, name_col], p_arab_loc)
        if result in ("Arab", "Jewish"):
            df.at[idx, output_col] = result
    return df

drug_cols = ['lsd', 'methamphetamine', 'ayahuasca', 'cathinone', 'ketamine',
             'hashish', 'methylmethcathinone', 'cannabis_plants', 'cannabis',
             'mdma', 'cocaine', 'heroin']
cannabis_group = ['cannabis', 'cannabis_plants']

def is_filled(val):
    return pd.notna(val) and str(val).strip() != ""

def split_drug(df, target):
    mask = df[target].apply(is_filled)
    for other_col in drug_cols:
        if other_col != target and not (target in cannabis_group and other_col in cannabis_group):
            mask = mask & ~df[other_col].apply(is_filled)
    df_only = df[mask].copy().rename(columns={target: 'total_amount'})
    return df_only.drop(columns=[c for c in drug_cols if c != target], errors='ignore')

CRIMINAL_RECORD_ORDER = {
    'ללא עבר פלילי': 0,
    'עבר פלילי קל':  1,
    'עבר פלילי כבד': 2,
}
CRIMINAL_RECORD_DECODE = {v: k for k, v in CRIMINAL_RECORD_ORDER.items()}

def impute_personal(df):
    # Same MICE + random forest imputation as the primary `impute` data
    df = df.reset_index(drop=True)
    df['age'] = pd.to_numeric(df['age'].replace('לא ידוע', np.nan), errors='coerce')

    df['marital_status'] = df['marital_status'].replace('לא ידוע', np.nan)
    status_categories = df['marital_status'].dropna().unique().tolist()
    status_to_code = {status: i for i, status in enumerate(status_categories)}
    code_to_status = {i: status for status, i in status_to_code.items()}
    df['marital_status_code'] = df['marital_status'].map(status_to_code)

    df['Criminal_record'] = df['Criminal_record'].replace('לא ידוע', np.nan)
    df['criminal_record_code'] = df['Criminal_record'].map(CRIMINAL_RECORD_ORDER)

    cols_to_impute = ['age', 'marital_status_code', 'criminal_record_code']
    helper_numeric = ['confession', 'responsabillity', 'regret', 'self_use', 'production_offence',
                      'use_offence', 'importation_offence', 'laboratory', 'sold_to_agent', 'year', 'total_amount']
    helper_numeric = [c for c in helper_numeric if c in df.columns]
    for col in helper_numeric:
        df[col] = pd.to_numeric(df[col].replace({'כן': 1, 'לא': 0}), errors='coerce')
    helper_categorical = ['district', 'court', 'ethnic', 'judge_ethnic']
    helper_categorical = [c for c in helper_categorical if c in df.columns]
    df_encoded = pd.get_dummies(df[helper_categorical], dummy_na=False)

    df_for_imputation = pd.concat([df[cols_to_impute], df[helper_numeric], df_encoded], axis=1)
    original_values = {col: df_for_imputation[col].copy()
                       for col in df_for_imputation.columns if col not in cols_to_impute}

    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=42),
        max_iter=10,
        random_state=42
    )
    df_imputed = pd.DataFrame(imputer.fit_transform(df_for_imputation), columns=df_for_imputation.columns)

    df['age'] = df_imputed['age'].round().astype(int)
    df['marital_status_code'] = df_imputed['marital_status_code'].round().astype(int).clip(0, len(status_categories) - 1)
    df['marital_status'] = df['marital_status_code'].map(code_to_status)
    df = df.drop(columns=['marital_status_code'])
    df['Criminal_record'] = df_imputed['criminal_record_code'].round().astype(int).clip(0, 2).map(CRIMINAL_RECORD_DECODE)
    df = df.drop(columns=['criminal_record_code'], errors='ignore')
    for col in original_values:
        if col in df.columns:
            df[col] = original_values[col]
    return df

before = pd.read_csv(rf"{BASE}\data_before_punishment.csv", encoding="utf-8-sig")
excluded = before[~before["original_case"].isin(features["original_case"])]
excluded = classify_ethnicity(excluded, "Defendants", "ethnic", features)
excluded = classify_ethnicity(excluded, "Judges", "judge_ethnic", features)
print(f"Excluded cases (no sentencing range): {len(excluded)}")
print(excluded["ethnic"].value_counts().to_string(), "\n")

for drug in DRUGS:
    kept = revert_imputation(pd.read_csv(rf"{BASE}\impute\{drug}.csv", encoding="utf-8-sig"))
    added = split_drug(excluded, drug).reindex(columns=kept.columns)
    df = impute_personal(pd.concat([kept, added], ignore_index=True))
    df.to_csv(rf"{BASE}\unfiltered\{drug}.csv", index=False, encoding="utf-8-sig")
    print(f"✅ {drug}: {len(kept)} filtered + {len(added)} excluded = {len(df)} rows")
    print(df["ethnic"].value_counts().to_string(), "\n")